# 🌍 Laboratório Estatístico Interativo
### Dataset: World Happiness Report (2005–presente)

Este notebook substitui o app Streamlit (que não estava renderizando corretamente no navegador). Ele usa `ipywidgets`, que é uma das opções de interface permitidas pelo enunciado.

## 📦 Módulo 0 — Carregar os dados

In [1]:
import sys
sys.path.append("..")  # garante que "src" seja encontrado

import math
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.minhastats import (
    media, mediana, moda, variancia, desvio_padrao,
    amplitude, quartis, coeficiente_variacao,
    covariancia, correlacao_pearson,
)

%matplotlib inline


In [2]:
# Carrega o CSV e normaliza os nomes das colunas
df = pd.read_csv("dados/world_happiness.csv")
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

print("Colunas encontradas:", list(df.columns))
print("Total de linhas:", len(df))
df.head(10)


Colunas encontradas: ['country_name', 'regional_indicator', 'year', 'life_ladder', 'log_gdp_per_capita', 'social_support', 'healthy_life_expectancy_at_birth', 'freedom_to_make_life_choices', 'generosity', 'perceptions_of_corruption', 'positive_affect', 'negative_affect', 'confidence_in_national_government']
Total de linhas: 2199


,country_name,regional_indicator,year,life_ladder,log_gdp_per_capita,social_support,healthy_life_expectancy_at_birth,freedom_to_make_life_choices,generosity,perceptions_of_corruption,positive_affect,negative_affect,confidence_in_national_government
0,Afghanistan,South Asia,2008,3.723590,7.350416,0.450662,50.500000,0.718114,0.167652,0.881686,0.414297,0.258195,0.612072
1,Afghanistan,South Asia,2009,4.401778,7.508646,0.552308,50.799999,0.678896,0.190809,0.850035,0.481421,0.237092,0.611545
2,Afghanistan,South Asia,2010,4.758381,7.613900,0.539075,51.099998,0.600127,0.121316,0.706766,0.516907,0.275324,0.299357
3,Afghanistan,South Asia,2011,3.831719,7.581259,0.521104,51.400002,0.495901,0.163571,0.731109,0.479835,0.267175,0.307386
4,Afghanistan,South Asia,2012,3.782938,7.660506,0.520637,51.700001,0.530935,0.237588,0.775620,0.613513,0.267919,0.435440
5,Afghanistan,South Asia,2013,3.572100,7.680333,0.483552,52.000000,0.577955,0.062666,0.823204,0.547417,0.273328,0.482847
6,Afghanistan,South Asia,2014,3.130896,7.670638,0.525568,52.299999,0.508514,0.105755,0.871242,0.491641,0.374861,0.409048
7,Afghanistan,South Asia,2015,3.982855,7.653833,0.528597,52.599998,0.388928,0.081652,0.880638,0.491410,0.339276,0.260557
8,Afghanistan,South Asia,2016,4.220169,7.650370,0.559072,52.924999,0.522566,0.043916,0.793246,0.501409,0.348332,0.324990
9,Afghanistan,South Asia,2017,2.661718,7.647830,0.490880,53.250000,0.427011,-0.119410,0.954393,0.435270,0.371326,0.261179


In [3]:
colunas_numericas = df.select_dtypes(include="number").columns.tolist()
colunas_categoricas = df.select_dtypes(exclude="number").columns.tolist()

print("Numéricas:", colunas_numericas)
print("Categóricas:", colunas_categoricas)


Numéricas: ['year', 'life_ladder', 'log_gdp_per_capita', 'social_support', 'healthy_life_expectancy_at_birth', 'freedom_to_make_life_choices', 'generosity', 'perceptions_of_corruption', 'positive_affect', 'negative_affect', 'confidence_in_national_government']
Categóricas: ['country_name', 'regional_indicator']


## 📊 Módulo 2 — Estatística Descritiva Interativa (variável numérica)

Escolha uma variável no menu abaixo. O notebook recalcula tudo automaticamente.

In [4]:
seletor_numerica = widgets.Dropdown(
    options=colunas_numericas,
    description="Variável:",
    style={"description_width": "initial"},
)

saida_numerica = widgets.Output()

def analisar_numerica(variavel):
    with saida_numerica:
        clear_output(wait=True)
        dados_variavel = df[variavel].dropna().tolist()

        print(f"=== {variavel} ===\n")
        print("--- Tendência central ---")
        print(f"Média:   {media(dados_variavel):.3f}")
        print(f"Mediana: {mediana(dados_variavel):.3f}")
        print(f"Moda:    {moda(dados_variavel)}")

        print("\n--- Dispersão ---")
        print(f"Amplitude:                {amplitude(dados_variavel):.3f}")
        print(f"Variância (amostral):     {variancia(dados_variavel):.3f}")
        print(f"Desvio padrão (amostral): {desvio_padrao(dados_variavel):.3f}")
        print(f"Coeficiente de variação:  {coeficiente_variacao(dados_variavel) * 100:.1f}%")

        q = quartis(dados_variavel)
        print("\n--- Quartis ---")
        print(f"Q1 (25%): {q['Q1']:.3f}")
        print(f"Q2 (50%): {q['Q2']:.3f}")
        print(f"Q3 (75%): {q['Q3']:.3f}")

        # Detecção de outliers pela regra do IQR
        iqr = q["Q3"] - q["Q1"]
        limite_inferior = q["Q1"] - 1.5 * iqr
        limite_superior = q["Q3"] + 1.5 * iqr
        outliers = [x for x in dados_variavel if x < limite_inferior or x > limite_superior]

        print("\n--- Outliers (regra do IQR) ---")
        print(f"Limites: [{limite_inferior:.3f}, {limite_superior:.3f}]")
        print(f"Quantidade de outliers: {len(outliers)}")

        # Interpretação automática da assimetria
        media_v = media(dados_variavel)
        mediana_v = mediana(dados_variavel)
        if abs(media_v - mediana_v) < 0.01 * (amplitude(dados_variavel) or 1):
            interpretacao = "A distribuição parece aproximadamente SIMÉTRICA (média ≈ mediana)."
        elif media_v > mediana_v:
            interpretacao = "A distribuição parece ter ASSIMETRIA À DIREITA (média > mediana)."
        else:
            interpretacao = "A distribuição parece ter ASSIMETRIA À ESQUERDA (média < mediana)."
        print("\n--- Interpretação ---")
        print(interpretacao)

        # Tabela de frequências (regra de Sturges)
        n = len(dados_variavel)
        k = max(1, math.ceil(1 + 3.322 * math.log10(n)))

        fig, axs = plt.subplots(1, 2, figsize=(12, 4))
        axs[0].hist(dados_variavel, bins=k, edgecolor="black")
        axs[0].set_title(f"Histograma de {variavel}")
        axs[0].set_xlabel(variavel)
        axs[0].set_ylabel("Frequência")

        axs[1].boxplot(dados_variavel, vert=False)
        axs[1].set_title(f"Boxplot de {variavel}")
        axs[1].set_xlabel(variavel)

        plt.tight_layout()
        plt.show()

def ao_mudar(mudanca):
    if mudanca["type"] == "change" and mudanca["name"] == "value":
        analisar_numerica(mudanca["new"])

seletor_numerica.observe(ao_mudar)
display(seletor_numerica, saida_numerica)
analisar_numerica(seletor_numerica.value)  # já mostra a primeira variável ao abrir


Dropdown(description='Variável:', options=('year', 'life_ladder', 'log_gdp_per_capita', 'social_support', 'hea…

Output()

## 📋 Estatística Descritiva (variável categórica)

In [5]:
seletor_categorica = widgets.Dropdown(
    options=colunas_categoricas,
    description="Variável:",
    style={"description_width": "initial"},
)

saida_categorica = widgets.Output()

def analisar_categorica(variavel):
    with saida_categorica:
        clear_output(wait=True)
        tabela_freq = df[variavel].value_counts().reset_index()
        tabela_freq.columns = [variavel, "Frequência"]
        display(tabela_freq.head(15))

        top15 = tabela_freq.head(15)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top15[variavel].astype(str), top15["Frequência"])
        ax.invert_yaxis()
        ax.set_title(f"Frequência de {variavel} (top 15)")
        plt.tight_layout()
        plt.show()

def ao_mudar_cat(mudanca):
    if mudanca["type"] == "change" and mudanca["name"] == "value":
        analisar_categorica(mudanca["new"])

seletor_categorica.observe(ao_mudar_cat)
display(seletor_categorica, saida_categorica)
analisar_categorica(seletor_categorica.value)


Dropdown(description='Variável:', options=('country_name', 'regional_indicator'), style=DescriptionStyle(descr…

Output()

## 🎲 Módulo 3 — Probabilidade e Simulação (Monte Carlo)

### (a) Lei dos Grandes Números

A ideia: quanto mais vezes você repetir um experimento aleatório (tipo jogar uma moeda), mais a frequência relativa observada se aproxima da probabilidade teórica real.

Aqui simulamos lançamentos de uma moeda (cara=1, coroa=0) e mostramos como a proporção de caras converge para 0,5 conforme o número de lançamentos cresce.

In [6]:
import random

slider_lancamentos = widgets.IntSlider(
    value=1000, min=10, max=20000, step=10,
    description="Nº lançamentos:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

saida_lgn = widgets.Output()

def simular_lgn(n_lancamentos):
    with saida_lgn:
        clear_output(wait=True)
        random.seed(42)  # reprodutibilidade
        lancamentos = [random.randint(0, 1) for _ in range(n_lancamentos)]

        # Frequência relativa acumulada de "cara" (1) a cada lançamento
        acumulado = 0
        frequencias_relativas = []
        for i, resultado in enumerate(lancamentos, start=1):
            acumulado += resultado
            frequencias_relativas.append(acumulado / i)

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(frequencias_relativas, color="steelblue")
        ax.axhline(0.5, color="red", linestyle="--", label="Probabilidade teórica (0,5)")
        ax.set_xlabel("Número de lançamentos")
        ax.set_ylabel("Frequência relativa de 'cara'")
        ax.set_title("Lei dos Grandes Números — convergência da frequência relativa")
        ax.legend()
        plt.tight_layout()
        plt.show()

        print(f"Frequência relativa final (após {n_lancamentos} lançamentos): {frequencias_relativas[-1]:.4f}")

def ao_mudar_lgn(mudanca):
    if mudanca["type"] == "change" and mudanca["name"] == "value":
        simular_lgn(mudanca["new"])

slider_lancamentos.observe(ao_mudar_lgn)
display(slider_lancamentos, saida_lgn)
simular_lgn(slider_lancamentos.value)


IntSlider(value=1000, description='Nº lançamentos:', layout=Layout(width='500px'), max=20000, min=10, step=10,…

Output()

### (b) Teorema Central do Limite (TCL)

Ideia: se você tirar várias amostras de qualquer variável do dataset (mesmo que ela não seja Normal) e calcular a média de cada amostra, a distribuição dessas médias vai se aproximar de uma distribuição Normal conforme o tamanho da amostra aumenta.

Escolha a variável, o número de amostras (repetições) e o tamanho de cada amostra.

In [7]:
seletor_var_tcl = widgets.Dropdown(
    options=colunas_numericas,
    description="Variável:",
    style={"description_width": "initial"},
)

slider_repeticoes = widgets.IntSlider(
    value=1000, min=100, max=5000, step=100,
    description="Nº de amostras:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

slider_tamanho_amostra = widgets.IntSlider(
    value=5, min=1, max=100, step=1,
    description="Tamanho de cada amostra:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

saida_tcl = widgets.Output()

def simular_tcl(variavel, n_repeticoes, tamanho_amostra):
    with saida_tcl:
        clear_output(wait=True)
        dados_variavel = df[variavel].dropna().tolist()
        random.seed(42)

        medias_amostrais = []
        for _ in range(n_repeticoes):
            amostra = random.choices(dados_variavel, k=tamanho_amostra)
            medias_amostrais.append(media(amostra))

        fig, axs = plt.subplots(1, 2, figsize=(12, 4))

        axs[0].hist(dados_variavel, bins=20, color="lightgray", edgecolor="black")
        axs[0].set_title(f"Distribuição ORIGINAL de {variavel}")

        axs[1].hist(medias_amostrais, bins=30, color="seagreen", edgecolor="black")
        axs[1].set_title(f"Distribuição das MÉDIAS AMOSTRAIS\n(n={tamanho_amostra}, repetições={n_repeticoes})")

        plt.tight_layout()
        plt.show()

        print(f"Média das médias amostrais: {media(medias_amostrais):.4f}")
        print(f"Média da variável original: {media(dados_variavel):.4f}")
        print(f"Desvio padrão das médias amostrais: {desvio_padrao(medias_amostrais):.4f}")
        print("\nRepare: quanto MAIOR o tamanho da amostra, mais a distribuição das médias")
        print("se parece com uma curva Normal (formato de sino), mesmo que os dados originais não sejam.")

def ao_mudar_tcl(mudanca=None):
    simular_tcl(seletor_var_tcl.value, slider_repeticoes.value, slider_tamanho_amostra.value)

seletor_var_tcl.observe(ao_mudar_tcl, names="value")
slider_repeticoes.observe(ao_mudar_tcl, names="value")
slider_tamanho_amostra.observe(ao_mudar_tcl, names="value")

display(seletor_var_tcl, slider_repeticoes, slider_tamanho_amostra, saida_tcl)
ao_mudar_tcl()


Dropdown(description='Variável:', options=('year', 'life_ladder', 'log_gdp_per_capita', 'social_support', 'hea…

IntSlider(value=1000, description='Nº de amostras:', layout=Layout(width='500px'), max=5000, min=100, step=100…

IntSlider(value=5, description='Tamanho de cada amostra:', layout=Layout(width='500px'), min=1, style=SliderSt…

Output()

## 📐 Módulo 4 — Distribuições Teóricas

Aqui comparamos o histograma real de uma variável com curvas de distribuições teóricas conhecidas (Normal, Uniforme, Exponencial), estimando os parâmetros de cada curva a partir dos próprios dados.

Isso ajuda a responder: *essa variável se comporta parecido com alguma distribuição clássica?*

In [8]:
from scipy import stats

seletor_var_dist = widgets.Dropdown(
    options=colunas_numericas,
    description="Variável:",
    style={"description_width": "initial"},
)

seletor_dist2 = widgets.Dropdown(
    options=["Uniforme", "Exponencial"],
    description="2ª distribuição (além da Normal):",
    style={"description_width": "initial"},
)

saida_dist = widgets.Output()

def comparar_distribuicoes(variavel, dist2):
    with saida_dist:
        clear_output(wait=True)
        dados_variavel = df[variavel].dropna().to_numpy()

        media_dados = media(dados_variavel.tolist())
        desvio_dados = desvio_padrao(dados_variavel.tolist())

        x = np.linspace(dados_variavel.min(), dados_variavel.max(), 300)

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.hist(dados_variavel, bins=25, density=True, color="lightgray",
                edgecolor="black", label="Dados reais")

        # Curva Normal, com média e desvio padrão estimados dos próprios dados
        curva_normal = stats.norm.pdf(x, loc=media_dados, scale=desvio_dados)
        ax.plot(x, curva_normal, color="crimson", linewidth=2,
                label=f"Normal (μ={media_dados:.2f}, σ={desvio_dados:.2f})")

        # Segunda distribuição escolhida pelo usuário
        if dist2 == "Uniforme":
            a, b = dados_variavel.min(), dados_variavel.max()
            curva2 = stats.uniform.pdf(x, loc=a, scale=(b - a))
            rotulo2 = f"Uniforme (a={a:.2f}, b={b:.2f})"
        else:  # Exponencial
            # A exponencial só faz sentido pra valores >= 0; deslocamos se precisar
            deslocamento = min(0, dados_variavel.min())
            escala = media_dados - deslocamento
            curva2 = stats.expon.pdf(x, loc=deslocamento, scale=escala)
            rotulo2 = f"Exponencial (λ=1/{escala:.2f})"

        ax.plot(x, curva2, color="steelblue", linewidth=2, linestyle="--", label=rotulo2)

        ax.set_title(f"Ajuste de distribuições teóricas — {variavel}")
        ax.set_xlabel(variavel)
        ax.set_ylabel("Densidade")
        ax.legend()
        plt.tight_layout()
        plt.show()

        print("Discussão automática do ajuste:")
        print(f"- A curva Normal foi estimada usando a média ({media_dados:.3f}) e o desvio padrão ({desvio_dados:.3f}) reais dos dados.")
        print(f"- Observe visualmente: quanto mais a barra cinza (dados reais) acompanha a linha vermelha,")
        print(f"  mais a variável '{variavel}' se aproxima de uma distribuição Normal.")
        print("- Isso é uma comparação visual (qualitativa); testes formais como Shapiro-Wilk")
        print("  poderiam confirmar estatisticamente o ajuste, mas fogem do escopo pedido aqui.")

import numpy as np

def ao_mudar_dist(mudanca=None):
    comparar_distribuicoes(seletor_var_dist.value, seletor_dist2.value)

seletor_var_dist.observe(ao_mudar_dist, names="value")
seletor_dist2.observe(ao_mudar_dist, names="value")

display(seletor_var_dist, seletor_dist2, saida_dist)
ao_mudar_dist()


Dropdown(description='Variável:', options=('year', 'life_ladder', 'log_gdp_per_capita', 'social_support', 'hea…

Dropdown(description='2ª distribuição (além da Normal):', options=('Uniforme', 'Exponencial'), style=Descripti…

Output()

## 📈 Módulo 5 — Correlação e Regressão Linear

Escolha uma variável **X** (independente/preditora) e uma variável **Y** (dependente/alvo). O notebook calcula, usando SUAS próprias funções de `minhastats.py`:

- Diagrama de dispersão
- Coeficiente de correlação de Pearson
- Reta de regressão (método dos mínimos quadrados, implementado na mão)
- Equação da reta e R²
- Um campo pra você digitar um valor de X e ver a previsão de Y

⚠️ **Lembrete importante**: correlação não implica causalidade!

In [9]:
seletor_x = widgets.Dropdown(
    options=colunas_numericas,
    value=colunas_numericas[0],
    description="Variável X:",
    style={"description_width": "initial"},
)

seletor_y = widgets.Dropdown(
    options=colunas_numericas,
    value=colunas_numericas[1] if len(colunas_numericas) > 1 else colunas_numericas[0],
    description="Variável Y:",
    style={"description_width": "initial"},
)

campo_predicao = widgets.FloatText(
    description="Prever Y para X =",
    style={"description_width": "initial"},
)

saida_regressao = widgets.Output()

# Guardamos os coeficientes calculados numa variável global simples,
# pra reaproveitar tanto no gráfico quanto na predição
coeficientes_atuais = {"b0": 0, "b1": 0}

def calcular_regressao(x_nome, y_nome):
    dados_x = df[x_nome].dropna()
    dados_y = df[y_nome].dropna()
    # Garante que usamos só as linhas onde AMBAS as colunas têm valor
    validos = pd.concat([dados_x, dados_y], axis=1).dropna()
    x = validos[x_nome].tolist()
    y = validos[y_nome].tolist()

    # Mínimos quadrados "na mão": b1 = covariância(x,y) / variância(x)
    b1 = covariancia(x, y) / variancia(x)
    b0 = media(y) - b1 * media(x)

    r = correlacao_pearson(x, y)
    r2 = r ** 2

    return x, y, b0, b1, r, r2

def atualizar_regressao(mudanca=None):
    with saida_regressao:
        clear_output(wait=True)
        x_nome, y_nome = seletor_x.value, seletor_y.value

        if x_nome == y_nome:
            print("Escolha duas variáveis DIFERENTES para X e Y.")
            return

        x, y, b0, b1, r, r2 = calcular_regressao(x_nome, y_nome)
        coeficientes_atuais["b0"] = b0
        coeficientes_atuais["b1"] = b1

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(x, y, alpha=0.4, color="steelblue", label="Dados reais")

        x_min, x_max = min(x), max(x)
        linha_x = [x_min, x_max]
        linha_y = [b0 + b1 * xi for xi in linha_x]
        ax.plot(linha_x, linha_y, color="crimson", linewidth=2, label="Reta de regressão")

        ax.set_xlabel(x_nome)
        ax.set_ylabel(y_nome)
        ax.set_title(f"{y_nome} em função de {x_nome}")
        ax.legend()
        plt.tight_layout()
        plt.show()

        sinal = "+" if b0 >= 0 else "-"
        print(f"Equação da reta:  {y_nome} = {b1:.4f} * {x_nome} {sinal} {abs(b0):.4f}")
        print(f"Coeficiente de correlação de Pearson (r): {r:.4f}")
        print(f"R² (coeficiente de determinação): {r2:.4f}")
        print(f"   -> Isso significa que {r2*100:.1f}% da variação em '{y_nome}'")
        print(f"      pode ser explicada linearmente por '{x_nome}'.")

        print("\nInterpretação do coeficiente b1:")
        print(f"A cada 1 unidade que '{x_nome}' aumenta, '{y_nome}' muda em média {b1:.4f} unidades.")

        print("\n⚠️  ATENÇÃO: correlação NÃO implica causalidade! Essa relação estatística")
        print("    não prova que uma variável CAUSA a outra — pode haver outros fatores envolvidos.")

seletor_x.observe(atualizar_regressao, names="value")
seletor_y.observe(atualizar_regressao, names="value")

display(seletor_x, seletor_y, saida_regressao)
atualizar_regressao()


Dropdown(description='Variável X:', options=('year', 'life_ladder', 'log_gdp_per_capita', 'social_support', 'h…

Dropdown(description='Variável Y:', index=1, options=('year', 'life_ladder', 'log_gdp_per_capita', 'social_sup…

Output()

In [10]:
saida_predicao = widgets.Output()

def prever(mudanca=None):
    with saida_predicao:
        clear_output(wait=True)
        x_valor = campo_predicao.value
        b0 = coeficientes_atuais["b0"]
        b1 = coeficientes_atuais["b1"]
        y_previsto = b0 + b1 * x_valor
        print(f"Para {seletor_x.value} = {x_valor}, a previsão é {seletor_y.value} ≈ {y_previsto:.4f}")

campo_predicao.observe(prever, names="value")
# Também recalcula a previsão automaticamente se X ou Y mudarem
seletor_x.observe(prever, names="value")
seletor_y.observe(prever, names="value")
display(campo_predicao, saida_predicao)
prever()


FloatText(value=0.0, description='Prever Y para X =', style=DescriptionStyle(description_width='initial'))

Output()

In [11]:
print("=" * 60)
print("1) PAÍSES MAIS E MENOS FELIZES (média de life_ladder)")
print("=" * 60)
media_por_pais = df.groupby("country_name")["life_ladder"].mean().sort_values()
print("\nTop 5 MAIS felizes:")
print(media_por_pais.tail(5))
print("\nTop 5 MENOS felizes:")
print(media_por_pais.head(5))

print("\n" + "=" * 60)
print("2) PAÍSES COM MAIOR MUDANÇA AO LONGO DO TEMPO")
print("=" * 60)
primeiro_ano = df.sort_values("year").groupby("country_name").first()["life_ladder"]
ultimo_ano = df.sort_values("year").groupby("country_name").last()["life_ladder"]
mudanca = (ultimo_ano - primeiro_ano).sort_values()
print("\nMAIOR QUEDA na felicidade (primeiro ano registrado vs. último):")
print(mudanca.head(5))
print("\nMAIOR AUMENTO na felicidade:")
print(mudanca.tail(5))

print("\n" + "=" * 60)
print("3) QUAL VARIÁVEL TEM MAIOR CORRELAÇÃO COM life_ladder?")
print("=" * 60)
for coluna in colunas_numericas:
    if coluna == "life_ladder":
        continue
    dados_x = df[coluna].dropna()
    dados_y = df["life_ladder"].dropna()
    validos = pd.concat([dados_x, dados_y], axis=1).dropna()
    if len(validos) > 10:
        r = correlacao_pearson(validos[coluna].tolist(), validos["life_ladder"].tolist())
        print(f"{coluna:35s} r = {r:.3f}")

print("\n" + "=" * 60)
print("4) BRASIL: como ele se compara à média mundial?")
print("=" * 60)
brasil = df[df["country_name"] == "Brazil"]["life_ladder"].dropna().tolist()
media_mundial = df["life_ladder"].dropna().tolist()
print(f"Média do Brasil: {media(brasil):.3f}")
print(f"Média mundial:   {media(media_mundial):.3f}")
print(f"Desvio padrão do Brasil ao longo dos anos: {desvio_padrao(brasil):.3f}")

1) PAÍSES MAIS E MENOS FELIZES (média de life_ladder)

Top 5 MAIS felizes:
country_name
Iceland        7.458607
Switzerland    7.474483
Norway         7.481820
Finland        7.619146
Denmark        7.673428
Name: life_ladder, dtype: float64

Top 5 MENOS felizes:
country_name
Afghanistan                 3.346632
South Sudan                 3.401875
Central African Republic    3.514954
Burundi                     3.548124
Rwanda                      3.654473
Name: life_ladder, dtype: float64

2) PAÍSES COM MAIOR MUDANÇA AO LONGO DO TEMPO

MAIOR QUEDA na felicidade (primeiro ano registrado vs. último):
country_name
Lebanon       -3.138818
Afghanistan   -2.442319
Jordan        -1.939054
Syria         -1.861419
Angola        -1.794163
Name: life_ladder, dtype: float64

MAIOR AUMENTO na felicidade:
country_name
Serbia                 1.494884
Bulgaria               1.534551
Georgia                1.617647
Nicaragua              1.932099
Congo (Brazzaville)    1.985126
Name: life_ladder, dty